# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Usaf007/flyrankai-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

**Ranked Queue and Reason Codes**

The raw probability output of our Random Forest model must be translated into clear directives for the content team. We filter the dataset for content with a high probability of position decay (>60%) and rank the resulting queue by `decay_probability` and historical `gsc_impressions` to prioritize high-value targets.

Each row is assigned a standardized `action_label` (e.g., `REVIEW_FOR_UPDATE`) and a `reason_code` (`HIGH_DECAY_PROBABILITY`) so human reviewers understand exactly why the model flagged the page.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import duckdb
import pandas as pd
import os
import json
from sklearn.ensemble import RandomForestClassifier
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')
con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}');")

rel = "hf://datasets/FlyRank/internship-warehouse"
fact_table = f"{rel}/fact_content_daily_performance/month=2026-03/*.parquet"
dim_table = f"{rel}/dim_content.parquet"

# Pull data and prepare the model
query = f"""
    SELECT
        f.content_hash_id,
        f.client_hash_id,
        f.report_date,
        f.gsc_impressions,
        f.ga4_sessions,
        f.gsc_clicks,
        date_diff('day', d.content_created_date, f.report_date) AS age_days,
        d.word_count,
        CAST(CASE WHEN f.gsc_avg_position > 10 THEN 1 ELSE 0 END AS INTEGER) AS target_is_declining
    FROM read_parquet('{fact_table}') f
    JOIN read_parquet('{dim_table}') d ON f.content_hash_id = d.content_hash_id
    WHERE f.ga4_data_available IS TRUE
    LIMIT 50000
"""
df = con.execute(query).df().dropna()

features = ['gsc_impressions', 'ga4_sessions', 'gsc_clicks', 'age_days', 'word_count']
X = df[features]
y = df['target_is_declining']

# Train the model and generate probabilities
rf = RandomForestClassifier(n_estimators=100, random_state=42).fit(X, y)
df['decay_probability'] = rf.predict_proba(X)[:, 1].round(4)

# Build the playbook queue
df_queue = df[df['decay_probability'] > 0.60].copy()
df_queue['action_label'] = 'REVIEW_FOR_UPDATE'
df_queue['reason_code'] = 'HIGH_DECAY_PROBABILITY'

# Sort by risk, then by historical volume to prioritize high-value pages
df_queue = df_queue.sort_values(by=['decay_probability', 'gsc_impressions'], ascending=[False, False])

display(df_queue[['content_hash_id', 'decay_probability', 'gsc_impressions', 'action_label', 'reason_code']].head(10))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,content_hash_id,decay_probability,gsc_impressions,action_label,reason_code
33572,content_3df3f32f3fd58dea,1.0,6887,REVIEW_FOR_UPDATE,HIGH_DECAY_PROBABILITY
49130,content_e8a52cf3d5988c07,1.0,6066,REVIEW_FOR_UPDATE,HIGH_DECAY_PROBABILITY
26205,content_bdf60c86117079be,1.0,5251,REVIEW_FOR_UPDATE,HIGH_DECAY_PROBABILITY
26449,content_573804af4f4fa09f,1.0,4635,REVIEW_FOR_UPDATE,HIGH_DECAY_PROBABILITY
33515,content_bdf60c86117079be,1.0,4410,REVIEW_FOR_UPDATE,HIGH_DECAY_PROBABILITY
28443,content_9d1e94cbd32b0a41,1.0,4371,REVIEW_FOR_UPDATE,HIGH_DECAY_PROBABILITY
26125,content_50b3f14cd04b531b,1.0,4129,REVIEW_FOR_UPDATE,HIGH_DECAY_PROBABILITY
27626,content_bdf60c86117079be,1.0,4031,REVIEW_FOR_UPDATE,HIGH_DECAY_PROBABILITY
14094,content_49267c758cdcb3a8,1.0,3989,REVIEW_FOR_UPDATE,HIGH_DECAY_PROBABILITY
39384,content_b88025fbf2493889,1.0,3989,REVIEW_FOR_UPDATE,HIGH_DECAY_PROBABILITY


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

**Intended Use & System Limits**
*   **Intended Use:** This playbook is a decision-support tool. It is designed to narrow down a massive catalog of client content into a focused, prioritized list of at-risk pages for human review.
*   **System Limits:** The model is observational, not causal. It identifies directional risk based on historical telemetry, but it cannot diagnose *why* a page is dropping (e.g., technical SEO failure vs. outdated content). Furthermore, it relies on concurrent metrics, meaning it is identifying pages that are actively beginning to decay, not forecasting months in advance.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Demonstrating how the playbook filters the noise for the content team
total_pages = len(df)
flagged_pages = len(df_queue)
reduction_pct = (1 - (flagged_pages / total_pages)) * 100

summary = pd.DataFrame({
    'Metric': ['Total Pages Evaluated', 'Pages Flagged for Review', 'Workload Reduction'],
    'Value': [total_pages, flagged_pages, f"{reduction_pct:.1f}%"]
})
display(summary)

,Metric,Value
0,Total Pages Evaluated,47764
1,Pages Flagged for Review,23661
2,Workload Reduction,50.5%


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

**Human-in-the-Loop & Exclusions**

Under no circumstances should this queue be used to automate content deletion or direct edits. A human must review the flagged pages against a strict no-go list.

**Do Not Alter:**
1.  **Legal & Compliance Documents:** Privacy policies or regulatory filings.
2.  **Highly Seasonal Pages:** Content that naturally loses traffic out of season (e.g., holiday guides).
3.  **New Content Volatility:** Pages younger than 30 days, as early ranking fluctuations are normal and do not represent permanent decay. We enforce this rule structurally below.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Structurally enforcing the no-go list for new content
initial_queue_size = len(df_queue)
df_queue = df_queue[df_queue['age_days'] >= 30]
filtered_out = initial_queue_size - len(df_queue)

display(pd.DataFrame({
    'Audit Action': ['Removed pages < 30 days old (Volatility Guard)'],
    'Rows Excluded': [filtered_out],
    'Final Queue Size': [len(df_queue)]
}))

,Audit Action,Rows Excluded,Final Queue Size
0,Removed pages < 30 days old (Volatility Guard),2459,21202


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

**Monitoring and Retraining Triggers**

To ensure the playbook remains reliable, we must monitor the model for data drift and decay in predictive power.
*   **Metric Degradation:** We will calculate and log the baseline ROC-AUC on the test set. If the trailing 30-day AUC drops below 0.80, the model must be retrained.
*   **External Triggers:** Any confirmed core algorithm update from major search engines should trigger an immediate manual audit of the feature distributions and a potential retrain.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

from sklearn.metrics import roc_auc_score

# Calculate baseline AUC to serve as our monitor threshold
baseline_auc = roc_auc_score(y, df['decay_probability'])

metrics = {
    "model_type": "RandomForestClassifier",
    "baseline_auc": round(baseline_auc, 4),
    "retrain_threshold_auc": 0.8000,
    "features_used": features
}

display(pd.DataFrame([metrics]))

,model_type,baseline_auc,retrain_threshold_auc,features_used
0,RandomForestClassifier,1.0,0.8,"[gsc_impressions, ga4_sessions, gsc_clicks, ag..."


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

**Exporting Final Artifacts**

We export the finalized, filtered priority queue to the `outputs` directory. This CSV will serve as the foundational data source for the Capstone research paper. We also export our baseline monitoring metrics as a receipt of our model's performance at the time of the playbook generation.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

os.makedirs('work/outputs', exist_ok=True)

# Export the prioritized queue
csv_path = 'work/outputs/action_playbook_queue.csv'
df_queue.to_csv(csv_path, index=False)

# Export the monitoring metrics receipt
json_path = 'work/outputs/metrics.json'
with open(json_path, 'w') as f:
    json.dump(metrics, f, indent=4)

display(pd.DataFrame({
    'File Exported': [csv_path, json_path],
    'Status': ['Success', 'Success']
}))


,File Exported,Status
0,work/outputs/action_playbook_queue.csv,Success
1,work/outputs/metrics.json,Success


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.